# HAGI — Stage 0 Training (Model A baseline)

Real training run on a Colab/Kaggle **T4** (or an Ampere GPU). The pipeline is
already validated by the smoke test. Run the cells **top to bottom**.

Checkpoints mirror to your private HuggingFace repo, so a killed session resumes
exactly where it stopped — just **re-run the notebook from the top** to continue.

**Before running: Runtime → Change runtime type → T4 GPU → Save.**


## 1. Clone the repo (experimental branch)


In [ ]:
import os
%cd /content
!test -d HAGI || git clone -b experimental https://github.com/ShmidtS/HAGI.git
%cd /content/HAGI
print('cwd:', os.getcwd())

## 2. Install dependencies (~3-5 min the first time)


In [ ]:
!pip install -q -r requirements.txt

## 3. HuggingFace token + checkpoint repo

Checkpoints are mirrored to a **private** HF repo for cross-session resume.

1. Get a **Write** token: huggingface.co/settings/tokens → New token → type *Write*.
2. Left sidebar **🔑 Secrets** → add `HF_TOKEN` = your token → enable **Notebook access**.
3. Edit `HF_REPO` below to `<your-hf-username>/hagi-stage0`.


In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
HF_REPO = 'YOUR_HF_USERNAME/hagi-stage0'   # <-- EDIT: your HuggingFace username
assert 'YOUR_HF_USERNAME' not in HF_REPO, 'Edit HF_REPO with your HF username first'
print('HF token loaded; checkpoints ->', HF_REPO)

## 4. Detect GPU and pick the config
T4 (16GB) → `stage0_t4.yaml` (fp16, seq 1024). Ampere 24GB+ → `baseline.yaml` (bf16).


In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'
p = torch.cuda.get_device_properties(0)
sm = p.major*10 + p.minor
CONFIG = 'configs/baseline.yaml' if (sm >= 80 and p.total_memory/1e9 >= 22) else 'configs/stage0_t4.yaml'
print(f'GPU: {p.name} | sm{sm} | {p.total_memory/1e9:.0f} GB -> {CONFIG}')

## 5. Tokenize the corpus to Drive (one-time, ~20 min)
Saved on Drive so it persists across sessions — you tokenize once, reuse forever.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/hagi-data'
if not os.path.isdir(DATA_DIR):
    !python -m prototype.data.tokenize --dataset HuggingFaceFW/fineweb-edu --subset sample-10BT \
        --output {DATA_DIR} --tokenizer HuggingFaceTB/SmolLM2-135M --limit 600000
else:
    print('shards already on Drive:', DATA_DIR)

## 6. Train one session
`--resume auto` pulls the latest checkpoint from your HF repo and continues.
`--steps 1500` ≈ ~50 min on a T4 — raise it for longer sessions. **Re-run this cell**
(or the whole notebook) each session; nothing is lost between runs.


In [ ]:
!python -m prototype.training.train --config {CONFIG} --data {DATA_DIR} \
    --device cuda --hf-repo {HF_REPO} --resume auto --steps 1500

## 7. (Optional) Evaluate once Stage 0 is trained
After ~1B tokens (~30,500 steps). Point `--ckpt` at a checkpoint under `checkpoints/stage0_t4/`.


In [ ]:
!ls checkpoints/stage0_t4/   # find your latest step-*.pt
# !python -m prototype.evaluation.evaluate --ckpt checkpoints/stage0_t4/step-00030000.pt \
#     --benchmarks arc_challenge,boolq --device cuda